# Post Table Hashtag Extraction and Preprocessing

This notebook performs the requested tasks:

1. Extract hashtags from `native_text` and `translated_text`.
2. Remove duplicate `post_id` rows.
3. Remove rows where both text columns are empty.
4. Convert `created_at` to datetime and extract `year`, `month`, and `day`.
5. Add original-text columns with hashtags removed: `native_text_no_hashtags` and `translated_text_no_hashtags`.
6. Clean text while keeping hashtag terms in `native_clean` and `translated_clean`.
7. Save the processed data to CSV.


In [19]:
import html
import re
import unicodedata
from pathlib import Path

import pandas as pd

## File Paths

In [20]:
input_path = Path('post_table_translation_fixed.csv')
output_path = Path('post_table_translation_preprocessed.csv')

input_path, output_path

(WindowsPath('post_table_translation_fixed.csv'),
 WindowsPath('post_table_translation_preprocessed.csv'))

## Helper Functions

In [21]:
URL_RE = re.compile(r'(?:https?://|www\.)\S+', flags=re.IGNORECASE)
MENTION_RE = re.compile(r'(?<!\w)@\w+', flags=re.UNICODE)
HTML_TAG_RE = re.compile(r'<[^>]+>')
WHITESPACE_RE = re.compile(r'\s+')


def is_hashtag_char(char):
    """Return True for Unicode letters, numbers, marks, and underscores."""
    category = unicodedata.category(char)
    return char == '_' or category[0] in {'L', 'M', 'N'}


def extract_hashtags(value, separator=' '):
    """Extract hashtags in original order, preserving the leading #."""
    if pd.isna(value):
        return ''

    text = str(value)
    hashtags = []
    index = 0

    while index < len(text):
        if text[index] != '#':
            index += 1
            continue

        start = index
        index += 1
        while index < len(text) and is_hashtag_char(text[index]):
            index += 1

        if index > start + 1:
            hashtags.append(text[start:index])

    return separator.join(hashtags)


def replace_hashtags(text, replacement=' '):
    """Replace hashtags using the same Unicode-aware logic as extraction."""
    result = []
    index = 0

    while index < len(text):
        if text[index] != '#':
            result.append(text[index])
            index += 1
            continue

        start = index
        index += 1
        while index < len(text) and is_hashtag_char(text[index]):
            index += 1

        if index == start + 1:
            result.append('#')
        else:
            result.append(replacement)

    return ''.join(result)


def convert_hashtags_to_words(text):
    """Convert #savewater to savewater so hashtags remain useful in clean text."""
    result = []
    index = 0

    while index < len(text):
        if text[index] != '#':
            result.append(text[index])
            index += 1
            continue

        start = index
        index += 1
        while index < len(text) and is_hashtag_char(text[index]):
            index += 1

        if index == start + 1:
            result.append('#')
        else:
            result.append(' ')
            result.append(text[start + 1:index])
            result.append(' ')

    return ''.join(result)


def remove_hashtags_from_original(value):
    """Keep the original post text, only removing hashtags and tidying leftover spaces."""
    if pd.isna(value):
        return ''

    text = replace_hashtags(str(value), replacement=' ')
    text = re.sub(r'[ \t]+([.,;:!?])', r'\1', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r' *\n *', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def remove_unicode_categories(text, remove_punctuation=True, remove_symbols=True):
    """Replace punctuation, symbols, and control characters with spaces."""
    chars = []

    for char in text:
        category = unicodedata.category(char)
        if category.startswith('C'):
            chars.append(' ')
        elif remove_punctuation and category.startswith('P'):
            chars.append(' ')
        elif remove_symbols and category.startswith('S'):
            chars.append(' ')
        else:
            chars.append(char)

    return ''.join(chars)


def clean_text(value, lowercase=True, remove_punctuation=True, remove_symbols=True, keep_hashtags=True):
    """Clean one text value while keeping hashtag terms by default."""
    if pd.isna(value):
        return ''

    text = html.unescape(str(value))
    text = unicodedata.normalize('NFKC', text)
    text = HTML_TAG_RE.sub(' ', text)
    text = URL_RE.sub(' ', text)
    text = MENTION_RE.sub(' ', text)

    if keep_hashtags:
        text = convert_hashtags_to_words(text)
    else:
        text = replace_hashtags(text, replacement=' ')

    text = remove_unicode_categories(
        text,
        remove_punctuation=remove_punctuation,
        remove_symbols=remove_symbols,
    )
    text = WHITESPACE_RE.sub(' ', text).strip()

    if lowercase:
        text = text.lower()

    return text


def text_is_empty(series):
    """Return True where a text column is null, empty, or whitespace only."""
    return series.isna() | series.astype(str).str.strip().eq('')


## Load Data

In [22]:
df = pd.read_csv(input_path)

print('Rows:', len(df))
print('Columns:', list(df.columns))
df.head()

Rows: 29352
Columns: ['post_id', 'utility_id', 'event_id', 'created_at', 'language', 'native_text', 'translated_text', 'reaction_count', 'comment_count', 'share_count', 'media_type', 'url']


,post_id,utility_id,event_id,created_at,language,native_text,translated_text,reaction_count,comment_count,share_count,media_type,url
0,1604452311683450,U10,NaN,2026-06-05 14:01:08+0000,Swedish,Växter gillar regn. \nSamla regnvatten i en tu...,Plants like rain. Collect rainwater in a barre...,15.0,0.0,0.0,image,https://www.facebook.com/stockholmvattenochavf...
1,1604170541711627,U10,NaN,2026-06-05 08:01:31+0000,Swedish,"Just nu är det högsäsong för att rycka, klippa...","Right now is the high season for pulling, cutt...",20.0,2.0,1.0,image,https://www.facebook.com/stockholmvattenochavf...
2,1599143538880994,U10,NaN,2026-05-31 06:01:17+0000,Swedish,Många undrar varför snittblommor inte får lägg...,Many wonder why cut flowers are not allowed in...,10.0,1.0,2.0,video,https://www.facebook.com/reel/989947877330615/
3,1414454017378592,U08,NaN,2026-05-31 11:45:36+0000,French,"🔴🔵 Hier soir, Paris a vibré avec le @psg et la...","🔴🔵 Last night, Paris vibrated with @psg and wa...",25.0,3.0,4.0,video,https://www.facebook.com/reel/1792060738428044/
4,1358540282969966,U08,NaN,2026-03-25 17:00:45+0000,French,Envie de bulles ? La nouvelle fontaine d’eau ...,Want bubbles? The new sparkling water fountain...,17.0,2.0,4.0,video,https://www.facebook.com/reel/1216862570233511/


## Task 1: Extract Hashtags

In [23]:
df['native_text_hashtags'] = df['native_text'].apply(extract_hashtags)
df['translated_text_hashtags'] = df['translated_text'].apply(extract_hashtags)

df[['native_text', 'native_text_hashtags', 'translated_text', 'translated_text_hashtags']].head()

,native_text,native_text_hashtags,translated_text,translated_text_hashtags
0,Växter gillar regn. \nSamla regnvatten i en tu...,#hållbarvattenanvändning #sparavatten,Plants like rain. Collect rainwater in a barre...,#sustainablewateruse #savewater
1,"Just nu är det högsäsong för att rycka, klippa...",,"Right now is the high season for pulling, cutt...",
2,Många undrar varför snittblommor inte får lägg...,,Many wonder why cut flowers are not allowed in...,
3,"🔴🔵 Hier soir, Paris a vibré avec le @psg et la...",,"🔴🔵 Last night, Paris vibrated with @psg and wa...",
4,Envie de bulles ? La nouvelle fontaine d’eau ...,,Want bubbles? The new sparkling water fountain...,


## Task 2: Preprocessing

In [24]:
original_rows = len(df)
duplicate_post_ids = df.duplicated(subset=['post_id']).sum()

df = df.drop_duplicates(subset=['post_id'], keep='first').copy()

both_text_empty = text_is_empty(df['native_text']) & text_is_empty(df['translated_text'])
empty_text_rows = both_text_empty.sum()

df = df.loc[~both_text_empty].copy()

print('Original rows:', original_rows)
print('Duplicate post_id rows removed:', duplicate_post_ids)
print('Rows removed where both text columns are empty:', empty_text_rows)
print('Rows remaining:', len(df))

Original rows: 29352
Duplicate post_id rows removed: 8469
Rows removed where both text columns are empty: 883
Rows remaining: 20000


## Convert Date and Extract Year, Month, Day

In [25]:
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce', utc=True)

df['year'] = df['created_at'].dt.year.astype('Int64')
df['month'] = df['created_at'].dt.month.astype('Int64')
df['day'] = df['created_at'].dt.day.astype('Int64')

print('created_at parse failures:', df['created_at'].isna().sum())
df[['created_at', 'year', 'month', 'day']].head()

created_at parse failures: 0


,created_at,year,month,day
0,2026-06-05 14:01:08+00:00,2026,6,5
1,2026-06-05 08:01:31+00:00,2026,6,5
2,2026-05-31 06:01:17+00:00,2026,5,31
3,2026-05-31 11:45:36+00:00,2026,5,31
4,2026-03-25 17:00:45+00:00,2026,3,25


## Add Original Text Without Hashtags and Clean Text Columns

In [26]:
df['native_text_no_hashtags'] = df['native_text'].apply(remove_hashtags_from_original)
df['translated_text_no_hashtags'] = df['translated_text'].apply(remove_hashtags_from_original)

# Hashtag terms are kept in the clean text, for example #savewater becomes savewater.
df['native_clean'] = df['native_text'].apply(clean_text)
df['translated_clean'] = df['translated_text'].apply(clean_text)

df[[
    'native_text_hashtags',
    'translated_text_hashtags',
    'native_text_no_hashtags',
    'translated_text_no_hashtags',
    'native_clean',
    'translated_clean',
]].head()


,native_text_hashtags,translated_text_hashtags,native_text_no_hashtags,translated_text_no_hashtags,native_clean,translated_clean
0,#hållbarvattenanvändning #sparavatten,#sustainablewateruse #savewater,Växter gillar regn.\nSamla regnvatten i en tun...,Plants like rain. Collect rainwater in a barre...,växter gillar regn samla regnvatten i en tunna...,plants like rain collect rainwater in a barrel...
1,,,"Just nu är det högsäsong för att rycka, klippa...","Right now is the high season for pulling, cutt...",just nu är det högsäsong för att rycka klippa ...,right now is the high season for pulling cutti...
2,,,Många undrar varför snittblommor inte får lägg...,Many wonder why cut flowers are not allowed in...,många undrar varför snittblommor inte får lägg...,many wonder why cut flowers are not allowed in...
3,,,"🔴🔵 Hier soir, Paris a vibré avec le @psg et la...","🔴🔵 Last night, Paris vibrated with @psg and wa...",hier soir paris a vibré avec le et la consomma...,last night paris vibrated with and water consu...
4,,,Envie de bulles? La nouvelle fontaine d’eau pé...,Want bubbles? The new sparkling water fountain...,envie de bulles la nouvelle fontaine d eau pét...,want bubbles the new sparkling water fountain ...


In [27]:
df['translated_text'][0]

'Plants like rain. Collect rainwater in a barrel and water gardens, plantings, and courtyards with a watering can — then you give the plants what they thrive on and save drinking water at the same time. Water early in the morning or late in the evening so the moisture stays longer in the soil and less water evaporates. 🌿 A small habit that makes a big difference. #sustainablewateruse #savewater'

In [28]:
df['translated_text_no_hashtags'][0]

'Plants like rain. Collect rainwater in a barrel and water gardens, plantings, and courtyards with a watering can — then you give the plants what they thrive on and save drinking water at the same time. Water early in the morning or late in the evening so the moisture stays longer in the soil and less water evaporates. 🌿 A small habit that makes a big difference.'

## Save Processed CSV

In [29]:
df.to_csv(output_path, index=False)
print(f'Saved cleaned file to: {output_path}')

Saved cleaned file to: post_table_translation_preprocessed.csv


## Verification

In [30]:
print('Rows:', len(df))
print('Duplicate post_id rows:', df.duplicated(subset=['post_id']).sum())
print('Rows where both text columns are empty:', (text_is_empty(df['native_text']) & text_is_empty(df['translated_text'])).sum())
print('Required new columns present:', all(
    col in df.columns
    for col in [
        'native_text_hashtags',
        'translated_text_hashtags',
        'native_text_no_hashtags',
        'translated_text_no_hashtags',
        'year',
        'month',
        'day',
        'native_clean',
        'translated_clean',
    ]
))

df[[
    'translated_text',
    'translated_text_hashtags',
    'translated_text_no_hashtags',
    'translated_clean',
]].head()


Rows: 20000
Duplicate post_id rows: 0
Rows where both text columns are empty: 0
Required new columns present: True


,translated_text,translated_text_hashtags,translated_text_no_hashtags,translated_clean
0,Plants like rain. Collect rainwater in a barre...,#sustainablewateruse #savewater,Plants like rain. Collect rainwater in a barre...,plants like rain collect rainwater in a barrel...
1,"Right now is the high season for pulling, cutt...",,"Right now is the high season for pulling, cutt...",right now is the high season for pulling cutti...
2,Many wonder why cut flowers are not allowed in...,,Many wonder why cut flowers are not allowed in...,many wonder why cut flowers are not allowed in...
3,"🔴🔵 Last night, Paris vibrated with @psg and wa...",,"🔴🔵 Last night, Paris vibrated with @psg and wa...",last night paris vibrated with and water consu...
4,Want bubbles? The new sparkling water fountain...,,Want bubbles? The new sparkling water fountain...,want bubbles the new sparkling water fountain ...


In [1]:
import pandas as pd
org = pd.read_csv('post_table_translation_preprocessed.csv')

In [2]:
org['translated_clean'][0]

'plants like rain collect rainwater in a barrel and water gardens plantings and courtyards with a watering can then you give the plants what they thrive on and save drinking water at the same time water early in the morning or late in the evening so the moisture stays longer in the soil and less water evaporates a small habit that makes a big difference sustainablewateruse savewater'